### formular una señal como una hipótesis falsable y distinguir predictibilidad de rentabilidad.

<b> Probabilidad/estadística </b> | ¿Tu señal realmente contiene información? Un researcher afirma: “cuando el retorno de los últimos 5 días es positivo, el siguiente día tiene mayor probabilidad de ser positivo”. 

En 1,000 observaciones encuentras 560 casos con momentum positivo; de éstos, 308 tuvieron retorno positivo al día siguiente. Calcula $(\hat p=308/560)$, plantea $(H_0:p=0.5)$ contra $(H_1:p>0.5)$ y calcula

$$ z=\frac{\hat p-0.5}{\sqrt{0.5(1-0.5)/560}}. $$

Decide aproximadamente si rechazarías $(H_0)$ al 5%. Después responde algo más importante que el p-value: aunque $(p>0.5)$ fuese estadísticamente convincente.

¿qué información todavía te falta para afirmar que existe una oportunidad de trading?
- Falta demostrar que la señal se traduce a PnL neto y no es un hallazgo accidental 
  - Magnitud economica: ver el retorno promedio/mediano condicionado a la señal y su distribucion. un 50.1%de aciertos puede perder dinero si las perdidas son mayores a las gancias 
  - costos de implementación: comisiones, spread, slippage 
  - regla operable : definir exactamente entrada, salida, tamaño de posicion, horarios, universo de activos, evitar look-ahead bias 
  - Riesgo : volatilidad, dradown, sharpe/sortino, exposicion a factores, comportamineto a crisis 
  - robustez : validacion fuera de muestra, walk-forward
  - data snooping: si se probo muchas señales, horizontes o umbrales, el p-value aislado deja de ser suficiente
  -estabilidad : verifica que el efecto no dependa de unos pocos dias, activos o regimenes de mercado

In [4]:
import numpy as np 
import scipy.stats as stats
p = 308/560
z_score = (p-0.5)/ (np.sqrt(0.5*(0.5) / 560))
p_value = stats.norm.sf(z_score) # sf : 1 - cdf ; mas preciso numericamente 
alfa = .05
if p_value < alfa: 
    print('Rechazamos H0')
else : 
    print('No rechazamos H0; No hay evidencia suficiente para rechazarlo')



Rechazamos H0


#### Código | Accuracy no es PnL

Simular 2,000 trades donde tu modelo acierta la dirección con probabilidad 0.56. 

Cuando acierta, genera +8 bps; cuando falla, -12 bps. Calcula accuracy, mean_return_bps, std_return_bps y PnL acumulado.

Después repite cambiando solamente el payoff a +12/-8 bps. Mantén aproximadamente la misma accuracy. 

Tu entregable debe ser una tabla pequeña comparando ambos experimentos. Explica en dos líneas por qué un clasificador con 56% de accuracy puede ser una estrategia mala y otro con exactamente la misma accuracy puede ser útil. Ésta es una conexión muy importante entre ML y trading que conviene tener automatizada mentalmente.

In [8]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(seed=42)

n_trades = 2_000
p_acierto = 0.56

# True = acierto; False = fallo
aciertos = rng.random(n_trades) < p_acierto

# Retorno por trade, en bps
retornos_bps = np.where(aciertos, 8, -12)

trades = pd.DataFrame({
    "acierto": aciertos,
    "retorno_bps": retornos_bps,
})

trades["pnl_acumulado_bps"] = trades["retorno_bps"].cumsum()
accuracy = trades["acierto"].mean()
mean_return_bps = trades["retorno_bps"].mean()
std_return_bps = trades["retorno_bps"].std()  # desviación estándar muestral

print(f"Accuracy: {accuracy:.2%}")
print(f"Mean return: {mean_return_bps:.2f} bps")
print(f"Std return: {std_return_bps:.2f} bps")

Accuracy: 55.85%
Mean return: -0.83 bps
Std return: 9.93 bps


In [6]:
print("Tasa de acierto:", trades["acierto"].mean())
print("PnL total:", trades["retorno_bps"].sum(), "bps")
print("PnL promedio por trade:", trades["retorno_bps"].mean(), "bps")

Tasa de acierto: 0.5585
PnL total: -1660 bps
PnL promedio por trade: -0.83 bps


In [7]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(seed=42)

n_trades = 2_000
p_acierto = 0.56

# True = acierto; False = fallo
aciertos = rng.random(n_trades) < p_acierto

# Retorno por trade, en bps
retornos_bps = np.where(aciertos, 12, -8)

trades = pd.DataFrame({
    "acierto": aciertos,
    "retorno_bps": retornos_bps,
})

trades["pnl_acumulado_bps"] = trades["retorno_bps"].cumsum()

trades.head()
print("Tasa de acierto:", trades["acierto"].mean())
print("PnL total:", trades["retorno_bps"].sum(), "bps")
print("PnL promedio por trade:", trades["retorno_bps"].mean(), "bps")

Tasa de acierto: 0.5585
PnL total: 6340 bps
PnL promedio por trade: 3.17 bps


La idea central es:
- Trading value $\neq$ Accuracy
- Trading value $ \approx$ expected return neto de costos, ajustado por riesgo

Un clasificador con 51% de accuracy puede ser excelente si selecciona bien los pocos movimientos grandes y limita las pérdidas. Y uno con 70% puede ser desastroso si cada error ocurre en un movimiento adverso muy grande.